In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
save_dir = "/content/drive/MyDrive/NLP_Project_Preprocessing"

In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 58.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pickle
from gensim.models import Word2Vec

In [ ]:
X_train_pad = np.load(f"{save_dir}/X_train_pad.npy")
X_test_pad = np.load(f"{save_dir}/X_test_pad.npy")

y_train_sentiment = np.load(f"{save_dir}/y_train_sentiment.npy")
y_test_sentiment = np.load(f"{save_dir}/y_test_sentiment.npy")

In [ ]:
embedding_matrix = np.load(
    f"{save_dir}/embedding_matrix.npy"
)

In [ ]:
w2v_model = Word2Vec.load(
    f"{save_dir}/word2vec.model"
)

In [ ]:
with open(f"{save_dir}/word_index.pkl", "rb") as f:
    word_index = pickle.load(f)

vocab_size = len(word_index) + 2

In [ ]:
import joblib

stance_encoder = joblib.load(
    f"{save_dir}/stance_encoder.pkl"
)

In [ ]:
print("X_train_pad:", X_train_pad.shape)
print("X_test_pad:", X_test_pad.shape)

print("y_train_sentiment:", y_train_sentiment.shape)
print("y_test_sentiment:", y_test_sentiment.shape)

print("Embedding Matrix:", embedding_matrix.shape)

print("Vocabulary Size:", len(word_index))

X_train_pad: (1166475, 30)
X_test_pad: (291619, 30)
y_train_sentiment: (1166475,)
y_test_sentiment: (291619,)
Embedding Matrix: (81958, 100)
Vocabulary Size: 81956


##model 1 computed class wts

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_sentiment)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_sentiment
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(0.8395355214264277), np.int64(1): np.float64(0.5681477790620028), np.int64(2): np.float64(20.508729363363045)}


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

lstm_model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=False
    ),

    LSTM(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [ ]:
history = lstm_model.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 76s 8ms/step - accuracy: 0.6739 - loss: 0.5925 - val_accuracy: 0.7424 - val_loss: 0.5666
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 63s 8ms/step - accuracy: 0.7271 - loss: 0.4957 - val_accuracy: 0.7427 - val_loss: 0.5619
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 62s 8ms/step - accuracy: 0.7448 - loss: 0.4613 - val_accuracy: 0.7558 - val_loss: 0.5404
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 81s 7ms/step - accuracy: 0.7557 - loss: 0.4381 - val_accuracy: 0.7809 - val_loss: 0.4860
Epoch 5/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 61s 7ms/step - accuracy: 0.7635 - loss: 0.4198 - val_accuracy: 0.7553 - val_loss: 0.5491
Epoch 6/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 62s 8ms/step - accuracy: 0.7693 - loss: 0.4054 - val_accuracy: 0.7919 - val_loss: 0.4579
Epoch 7/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 60s 7ms/step - accuracy: 0.7730 - loss: 0.3928 - val_accuracy: 0.7647 - val_loss: 0.5124
Epoch 8/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 61s 7ms/step - accuracy: 0.7764 - loss: 0

In [ ]:
test_loss, test_acc = lstm_model.evaluate(
    X_test_pad,
    y_test_sentiment,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 49s 5ms/step - accuracy: 0.7913 - loss: 0.4584
Test Accuracy: 0.7912790179252625


In [ ]:
import numpy as np

y_pred_probs = lstm_model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 24s 3ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.78      0.81      0.79    115785
     Neutral       0.86      0.78      0.82    171094
    Positive       0.24      0.84      0.37      4740

    accuracy                           0.79    291619
   macro avg       0.63      0.81      0.66    291619
weighted avg       0.82      0.79      0.80    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[ 93249  21196   1340]
 [ 26157 133519  11418]
 [    91    665   3984]]


#model 2 class wts + trainable embeddings

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_sentiment)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_sentiment
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(0.8395355214264277), np.int64(1): np.float64(0.5681477790620028), np.int64(2): np.float64(20.508729363363045)}


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

lstm_model_2 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    LSTM(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
lstm_model_2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = lstm_model_2.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 101s 12ms/step - accuracy: 0.7348 - loss: 0.4975 - val_accuracy: 0.7771 - val_loss: 0.4951
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 93s 11ms/step - accuracy: 0.8009 - loss: 0.3587 - val_accuracy: 0.7947 - val_loss: 0.4540
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 109s 13ms/step - accuracy: 0.8263 - loss: 0.3075 - val_accuracy: 0.8069 - val_loss: 0.4356
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 93s 11ms/step - accuracy: 0.8435 - loss: 0.2738 - val_accuracy: 0.8024 - val_loss: 0.4532
Epoch 5/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 93s 11ms/step - accuracy: 0.8582 - loss: 0.2474 - val_accuracy: 0.8109 - val_loss: 0.4540


In [ ]:
test_loss, test_acc = lstm_model_2.evaluate(
    X_test_pad,
    y_test_sentiment,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 41s 5ms/step - accuracy: 0.8071 - loss: 0.4336
Test Accuracy: 0.8070667386054993


In [ ]:
import numpy as np

y_pred_probs = lstm_model_2.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 25s 3ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.78      0.87      0.82    115785
     Neutral       0.90      0.76      0.83    171094
    Positive       0.24      0.89      0.38      4740

    accuracy                           0.81    291619
   macro avg       0.64      0.84      0.67    291619
weighted avg       0.84      0.81      0.82    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)
print(cm)

[[100437  14491    857]
 [ 27776 130707  12611]
 [    97    431   4212]]


##model 3 with wts= 1,0.7,10 and trainable=true

In [ ]:
class_weights = {
    0: 1.0,
    1: 0.7,
    2: 10.0
}

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

lstm_model_3 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    LSTM(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
lstm_model_3.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = lstm_model_3.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 96s 11ms/step - accuracy: 0.7755 - loss: 0.4820 - val_accuracy: 0.8286 - val_loss: 0.3975
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 93s 11ms/step - accuracy: 0.8284 - loss: 0.3620 - val_accuracy: 0.8295 - val_loss: 0.3824
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 94s 11ms/step - accuracy: 0.8486 - loss: 0.3167 - val_accuracy: 0.8226 - val_loss: 0.4074
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 96s 12ms/step - accuracy: 0.8640 - loss: 0.2833 - val_accuracy: 0.8244 - val_loss: 0.4141


In [ ]:
test_loss, test_acc = lstm_model_3.evaluate(
    X_test_pad,
    y_test_sentiment,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 42s 5ms/step - accuracy: 0.8284 - loss: 0.3841
Test Accuracy: 0.828392505645752


In [ ]:
import numpy as np

y_pred_probs = lstm_model_3.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 24s 3ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.82      0.83      0.82    115785
     Neutral       0.88      0.83      0.85    171094
    Positive       0.31      0.84      0.45      4740

    accuracy                           0.83    291619
   macro avg       0.67      0.83      0.71    291619
weighted avg       0.84      0.83      0.83    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[ 96122  18902    761]
 [ 21430 141470   8194]
 [    77    680   3983]]


##model 4 (0.9,0.7,14) + trainable = true

In [ ]:
class_weights = {
    0: 0.9,
    1: 0.7,
    2: 14.0
}

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

lstm_model_4 = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    LSTM(64),

    Dropout(0.5),

    Dense(32, activation="relu"),

    Dense(3, activation="softmax")
])

In [ ]:
lstm_model_4.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = lstm_model_4.fit(
    X_train_pad,
    y_train_sentiment,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 101s 12ms/step - accuracy: 0.7624 - loss: 0.5094 - val_accuracy: 0.7981 - val_loss: 0.4534
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 94s 11ms/step - accuracy: 0.8210 - loss: 0.3729 - val_accuracy: 0.8108 - val_loss: 0.4148


In [ ]:
test_loss, test_acc = lstm_model_4.evaluate(
    X_test_pad,
    y_test_sentiment,
    verbose=1
)

print("Test Accuracy:", test_acc)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 41s 4ms/step - accuracy: 0.7965 - loss: 0.4552
Test Accuracy: 0.7965153455734253


In [ ]:
import numpy as np

y_pred_probs = lstm_model_4.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 24s 3ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.80      0.82      0.81    115785
     Neutral       0.87      0.78      0.82    171094
    Positive       0.21      0.90      0.34      4740

    accuracy                           0.80    291619
   macro avg       0.63      0.83      0.66    291619
weighted avg       0.83      0.80      0.81    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[ 95227  19063   1495]
 [ 23654 132796  14644]
 [    67    417   4256]]
